In [1]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from recommender import build_user_item_matrix, train_svd_model
from evaluation import rmse, precision_at_k

ratings = pd.read_csv(
    "../data/u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"]
)

train_ratings, test_ratings = train_test_split(ratings, test_size=0.2, random_state=42)
print("Train size:", train_ratings.shape, "Test size:", test_ratings.shape)

Train size: (80000, 4) Test size: (20000, 4)


In [2]:
train_matrix = build_user_item_matrix(train_ratings)
reconstructed_df, svd_model = train_svd_model(train_matrix, n_components=20)

y_true = []
y_pred = []

for _, row in test_ratings.iterrows():
    user_id = row["user_id"]
    item_id = row["item_id"]
    actual_rating = row["rating"]

    if user_id in reconstructed_df.index and item_id in reconstructed_df.columns:
        predicted_rating = reconstructed_df.loc[user_id, item_id]
        y_true.append(actual_rating)
        y_pred.append(predicted_rating)

test_rmse = rmse(np.array(y_true), np.array(y_pred))
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Evaluated on {len(y_true)} test ratings")

Test RMSE: 2.6401
Evaluated on 19969 test ratings


In [3]:
global_mean = train_ratings["rating"].mean()
baseline_preds = np.full(len(test_ratings), global_mean)
baseline_rmse = rmse(test_ratings["rating"].values, baseline_preds)
print(f"Naive baseline RMSE (always predict global mean {global_mean:.2f}): {baseline_rmse:.4f}")

Naive baseline RMSE (always predict global mean 3.53): 1.1239


In [4]:
from recommender import train_svd_model_centered

reconstructed_centered, svd_model_centered, user_means = train_svd_model_centered(train_matrix, n_components=20)

y_true_c = []
y_pred_c = []

for _, row in test_ratings.iterrows():
    user_id = row["user_id"]
    item_id = row["item_id"]
    actual_rating = row["rating"]

    if user_id in reconstructed_centered.index and item_id in reconstructed_centered.columns:
        predicted_rating = reconstructed_centered.loc[user_id, item_id]
        y_true_c.append(actual_rating)
        y_pred_c.append(predicted_rating)

test_rmse_centered = rmse(np.array(y_true_c), np.array(y_pred_c))
print(f"Test RMSE (mean-centered SVD): {test_rmse_centered:.4f}")

Test RMSE (mean-centered SVD): 0.9888


In [5]:
def get_top_n_for_user(user_id, reconstructed_df, original_matrix, n=10):
    user_predictions = reconstructed_df.loc[user_id]
    already_rated = original_matrix.loc[user_id]
    unrated_items = user_predictions[already_rated == 0]
    return unrated_items.sort_values(ascending=False).head(n).index.tolist()

precisions = []
test_users = test_ratings["user_id"].unique()[:100]  # sample 100 users for speed

for user_id in test_users:
    if user_id not in reconstructed_centered.index:
        continue
    relevant_items = set(test_ratings[(test_ratings["user_id"] == user_id) & (test_ratings["rating"] >= 4)]["item_id"])
    if len(relevant_items) == 0:
        continue
    recommended = get_top_n_for_user(user_id, reconstructed_centered, train_matrix, n=10)
    precisions.append(precision_at_k(recommended, relevant_items, k=10))

print(f"Average Precision@10 across {len(precisions)} users: {np.mean(precisions):.4f}")

Average Precision@10 across 100 users: 0.2850
